# FSPS F115W Mstar-to-flux ratios

Observed-frame JWST/NIRCam F115W conversion for WFI2033 lens redshift `z_lens=0.6575`.

Local FSPS setup used here:

- `SPS_HOME=/home/skylee/fsps`
- conda env: `/home/skylee/anaconda3/envs/jwst_env`
- `fsps==0.4.7`

Summary copied to `paper.ipynb`:

| IMF | min [Msun/nJy] | median [Msun/nJy] | max [Msun/nJy] |
| --- | ---: | ---: | ---: |
| Chabrier | 1.6579368e6 | 2.7324340e6 | 4.0218466e6 |
| Salpeter | 3.0763298e6 | 5.0710854e6 | 7.4868236e6 |

In [ ]:
import os
os.environ.setdefault('SPS_HOME', '/home/skylee/fsps')

import fsps
import numpy as np
import pandas as pd

In [ ]:
Z_LENS = 0.6575
BAND = 'jwst_f115w'
IMF_TYPES = {'salpeter': 0, 'chabrier': 1}

age_grid_gyr = np.linspace(3.0, 7.0, 9)
logzsol_grid = [-0.2, 0.0, 0.2]


def abmag_to_njy(m_ab):
    return 10**((31.4 - m_ab) / 2.5)


def make_population(imf, logzsol):
    sp = fsps.StellarPopulation(
        zcontinuous=1,
        imf_type=IMF_TYPES[imf],
        logzsol=logzsol,
        sfh=0,
        compute_vega_mags=False,
        add_neb_emission=False,
        add_stellar_remnants=True,
    )
    sp.params['dust_type'] = 2
    sp.params['dust2'] = 0.0
    return sp

In [ ]:
rows = []

for imf in ['chabrier', 'salpeter']:
    for logzsol in logzsol_grid:
        sp = make_population(imf, logzsol)
        for age_gyr in age_grid_gyr:
            sp.get_spectrum(tage=age_gyr)
            m_ab = sp.get_mags(tage=age_gyr, redshift=Z_LENS, bands=[BAND])[0]
            f_njy = abmag_to_njy(m_ab)
            mstar_per_njy = sp.stellar_mass / f_njy
            rows.append({
                'imf': imf,
                'age_gyr': age_gyr,
                'logzsol': logzsol,
                'm_ab': m_ab,
                'f_njy': f_njy,
                'stellar_mass': sp.stellar_mass,
                'mstar_per_njy': mstar_per_njy,
            })

fsps_grid = pd.DataFrame(rows)
fsps_grid

In [ ]:
imf_ml_summary = (
    fsps_grid
    .groupby('imf')['mstar_per_njy']
    .agg(['min', 'median', 'max'])
)

imf_ml_summary

In [ ]:
imf_ml_ranges = {
    imf: {
        'min': row['min'],
        'median': row['median'],
        'max': row['max'],
        'unit': 'Msun / nJy',
    }
    for imf, row in imf_ml_summary.iterrows()
}

imf_ml_ranges